# Cloudflare Issue Tracking ETL Pipeline - Executive Walkthrough

This notebook demonstrates the end-to-end functionality of our AI-driven Competitive Intelligence pipeline. The goal of this pipeline is to gather external developer friction points from community forums, score their relevance against Cloudflare products via LLMs, and generate simulated user prompts for sentiment analysis.

### Initial Setup and Configuration

In [ ]:
import pandas as pd
import yaml
import json

# Load our configuration (APIs, Prompt Templates, Storage mappings)
with open('config/settings.yaml', 'r') as f:
    config = yaml.safe_load(f)
    
print("Pipeline Configuration Loaded.")
print(f"Configured LLM Provider: {config['llm']['provider'].upper()} ({config['llm']['model']})")

### Phase 1: Data Ingestion
We pull raw issues across a wide digital landscape. In this demonstration, we are utilizing the module stubs which simulate hitting the GitHub, StackOverflow, and Reddit APIs without needing live rate-limited credentials.

In [ ]:
from src.fetchers.github import fetch_github_issues
from src.fetchers.stackoverflow import fetch_stackoverflow_questions
from src.fetchers.reddit import fetch_reddit_posts

issues = []
issues.extend(fetch_github_issues(config))
issues.extend(fetch_stackoverflow_questions(config))
issues.extend(fetch_reddit_posts(config))

print(f"\nTotal cross-platform issues fetched successfully: {len(issues)}")

### Phases 2 & 3: AI Inference (Relevance & Sentiment Mapping)
This step pushes the raw issues through the configured prompt templates into the LLM logic layer. It will only further analyze issues that are marked relevant to Cloudflare.

In [ ]:
from src.analyzers.relevance_classifier import classify_issue_relevance
from src.analyzers.sentiment_authority import analyze_sentiment_and_authority

fully_analyzed_issues = []

for issue in issues:
    processed = classify_issue_relevance(issue, config)
    
    # If the LLM determines Cloudflare can solve it, extract deeper insight
    if processed.get('relevance', {}).get('relevant'):
        analyzed = analyze_sentiment_and_authority(processed, config)
        fully_analyzed_issues.append(analyzed)

print(f"{len(fully_analyzed_issues)} highly relevant issues parsed out from the noise.")

### Phase 4: Reporting and Output Generation
We use `pandas` to flatten the data payload and extract key metrics into two powerful artifacts:
1. **Processed Issues Database** (Comprehensive issue spreadsheet)
2. **Simulated User Prompts** (A map of generated prompts representing how developers might query AI assistants about these friction points)

In [ ]:
from src.data_writer import save_analyzed_data
import os

# Override the config explicitly to output into this exact folder temporarily for demonstration
config['storage']['output_dir'] = './data'

save_analyzed_data(fully_analyzed_issues, config)
print("\nStorage process complete.")

### View the Master Issues Data
This output provides product managers and sales teams a structured, data-driven view of what's happening.

In [ ]:
if os.path.exists('./data/processed_issues.csv'):
    df = pd.read_csv('./data/processed_issues.csv')
    display(df[['source', 'title', 'relevance.confidence_score', 'analysis.sentiment.cloudflare']])
else:
    print("File not found.")

### View Simulated User Prompts
This output provides simulated generative AI prompts that developers might naturally write to explore these issues, formatted for later sentiment analysis models.

In [ ]:
if os.path.exists('./data/simulated_prompts.json'):
    with open('./data/simulated_prompts.json', 'r') as f:
        simulated_prompts = json.load(f)

    print("Simulated User Prompts for Sentiment Analysis:\n")
    print(json.dumps(simulated_prompts, indent=2))
else:
    print("File not found.")